# 4단계 — 급추가 자료 정리 (pandas 심화 / numpy / datetime / 인코딩)

시험 범위인지 확실친 않지만, 급하게 추가된 자료라 혹시 몰라 노트북 형태로도 정리했습니다.
직접 셀을 실행해보면서 익히시면 됩니다. (일부 "종합 실습" 코드는 원본 CSV 파일이 있어야 실행되므로 참고용 마크다운으로만 남겨뒀습니다.)

**목차**
1. Pandas 심화 — merge / concat / str / explode / apply·map / pivot_table
2. Numpy — ndarray / 벡터연산 / 브로드캐스팅 / 불리언마스크 / axis / nan
3. datetime — strptime·strftime / to_datetime / .dt / 나이계산 / 기간집계
4. 인코딩과 디코딩 — str·bytes / encode·decode / base64 / URL인코딩 / CSV·requests 인코딩
5. 예상문제 (사지선다)


## 1. Pandas 심화

### 1-1. `merge()` — 두 표를 옆으로 합치기 (엑셀 VLOOKUP과 비슷)

`how` 옵션이 핵심입니다.

| how | 의미 |
|---|---|
| `'inner'` | 양쪽 **모두에 있는** key만 (기본값) |
| `'left'` | 왼쪽 표는 **전부 남기고**, 없으면 NaN |
| `'right'` | 오른쪽 표를 전부 남김 |
| `'outer'` | 양쪽 전부 남김 |

In [ ]:
import pandas as pd
import numpy as np

members = pd.DataFrame({
    'name':     ['지민', '수진', '태양', '하늘'],
    'group_id': ['G1', 'G1', 'G2', 'G9'],   # G9 는 groups 에 없는 값
})

groups = pd.DataFrame({
    'group_id': ['G1', 'G2', 'G3'],
    'company':  ['하이브', 'SM', 'JYP'],
})

print('=== how="inner" (기본값) ===')
print(pd.merge(members, groups, on='group_id', how='inner'))

print()
print('=== how="left" ===')
print(pd.merge(members, groups, on='group_id', how='left'))

In [ ]:
# 합친 뒤 결측 확인 = "짝을 못 찾은 행이 몇 개인가" 점검
merged = pd.merge(members, groups, on='group_id', how='left')

print('company 결측 개수:', merged['company'].isna().sum())
print('짝을 못 찾은 행:')
print(merged[merged['company'].isna()])

# 기준 열의 이름이 서로 다르면 left_on / right_on 사용
# pd.merge(members, groups, left_on='group_id', right_on='gid', how='left')

### 1-2. `concat()` — 표를 위아래로 이어붙이기

`merge`가 **옆으로**(열 추가)라면, `concat`은 **아래로**(행 추가)입니다. 스크래핑에서 1페이지, 2페이지 결과를 하나로 합칠 때 씁니다.

In [ ]:
page1 = pd.DataFrame({'title': ['A', 'B'], 'price': [1000, 2000]})
page2 = pd.DataFrame({'title': ['C', 'D'], 'price': [3000, 4000]})

# ignore_index=True : 인덱스를 0,1,2,3 으로 새로 매긴다 (빼면 0,1,0,1 처럼 중복됨)
all_pages = pd.concat([page1, page2], ignore_index=True)
print(all_pages)

print()
print('--- ignore_index 를 빼면 ---')
print(pd.concat([page1, page2]))

In [ ]:
# 실전 패턴: 반복문으로 페이지를 모아 리스트에 담고, 마지막에 한 번만 concat
frames = []
for page in range(1, 4):
    # 실제로는 여기서 requests + BeautifulSoup 로 스크래핑
    one = pd.DataFrame({'page': [page] * 2, 'item': [f'item{page}-1', f'item{page}-2']})
    frames.append(one)

result = pd.concat(frames, ignore_index=True)
print(result)
print('총', len(result), '행')

### 1-3. `.str` 접근자 — 문자열 열을 한 번에 가공

문자열이 든 열에 `.str`을 붙이면 **모든 행에 문자열 메서드가 한 번에 적용**됩니다. 스크래핑한 데이터는 공백·단위·기호가 섞여 있으므로 이 정제 과정이 거의 항상 필요합니다.

In [ ]:
raw = pd.DataFrame({
    'name':  ['  아이유  ', '방탄소년단', '  블랙핑크'],
    'genre': ['Ballad, Pop', 'Hip-Hop, Pop', 'Dance, Pop'],
    'price': ['25,000원', '30,000원', '28,000원'],
})

# strip()   : 앞뒤 공백 제거
raw['name'] = raw['name'].str.strip()

# replace() : 문자 치환 -> 숫자로 변환
raw['price_num'] = raw['price'].str.replace(',', '').str.replace('원', '').astype(int)

print(raw)
print('가격 평균:', raw['price_num'].mean())

In [ ]:
# contains() : 특정 문자열 포함 여부 -> 필터링에 사용
print('Pop 이 들어간 행:')
print(raw[raw['genre'].str.contains('Pop')])

In [ ]:
# split() : 쪼개기. expand=True 면 여러 열로 펼쳐진다
print('첫 번째 장르만 추출:')
print(raw['genre'].str.split(', ').str[0])

print()
print('여러 열로 펼치기 (expand=True):')
print(raw['genre'].str.split(', ', expand=True))

### 1-4. `explode()` — 한 칸에 여러 값이 든 데이터 펼치기

`"Drama, Comedy, Action"`처럼 한 칸에 여러 값이 쉼표로 묶여 있으면 집계를 할 수 없습니다.
`.str.split()`으로 리스트를 만들고 `.explode()`로 한 값당 한 행으로 펼칩니다.

In [ ]:
movies = pd.DataFrame({
    'title':     ['영화A', '영화B'],
    'listed_in': ['Drama, Comedy', 'Action, Drama, Thriller'],
})

# 1단계: 문자열 -> 리스트
movies['genre'] = movies['listed_in'].str.split(', ')
print('--- split 후 (한 칸에 리스트가 들어있다) ---')
print(movies)

In [ ]:
# 2단계: 리스트 -> 행으로 펼치기
exploded = movies.explode('genre')
print('--- explode 후 (한 장르당 한 행) ---')
print(exploded)

# 펼친 뒤에야 value_counts() 로 장르별 개수를 셀 수 있다
print()
print('장르별 작품 수:')
print(exploded['genre'].value_counts())

# 인덱스가 0,0,1,1,1 로 중복된 점에 주의 -> reset_index(drop=True) 로 새로 매길 수 있다
exploded = exploded.reset_index(drop=True)
print()
print(exploded)

### 1-5. `apply()` / `Series.map()` — 내가 만든 함수를 적용

- `Series.apply(함수)`: 각 **값**에 함수 적용
- `DataFrame.apply(함수, axis=1)`: 각 **행 전체**에 함수 적용 (여러 열을 같이 봐야 할 때)

In [ ]:
books = pd.DataFrame({
    'title': ['파이썬 입문', '데이터 분석', '웹 스크래핑', '머신러닝'],
    'price': [25000, 30000, 28000, 33000],
    'pages': [300, 450, 380, 520],
})


def price_grade(price):
    # 가격을 등급 문자열로 변환한다.
    if price < 27000:
        return '저가'
    elif price < 31000:
        return '중가'
    return '고가'


# Series.apply : price 열의 각 '값' 에 함수를 적용
books['grade'] = books['price'].apply(price_grade)
print(books)

In [ ]:
# DataFrame.apply(axis=1) : 각 '행' 을 통째로 받아 여러 열을 함께 사용
books['price_per_page'] = books.apply(
    lambda row: round(row['price'] / row['pages'], 1), axis=1
)
print(books[['title', 'price', 'pages', 'price_per_page']])

# 간단한 값 치환은 map(딕셔너리) 가 더 빠르고 읽기 쉽다
grade_emoji = {'저가': '저', '중가': '중', '고가': '고'}
books['icon'] = books['grade'].map(grade_emoji)
print()
print(books[['title', 'grade', 'icon']])

### 1-6. `pivot_table()` — 교차표로 요약

`groupby`가 **한 방향 집계**라면, `pivot_table`은 **행·열 두 방향 교차표**를 만듭니다.

In [ ]:
sales = pd.DataFrame({
    'year':    [2022, 2022, 2022, 2023, 2023, 2023, 2024],
    'type':    ['Movie', 'TV', 'TV', 'Movie', 'TV', 'Movie', 'TV'],
    'title':   ['a', 'b', 'b2', 'c', 'd', 'e', 'f'],
    'revenue': [100, 200, 150, 250, 120, 300, 180],
})

# groupby: 결과가 세로로 길게 나온다
print('=== groupby ===')
print(sales.groupby(['year', 'type'])['revenue'].sum())

In [ ]:
# pivot_table: 같은 내용이 표(교차표) 형태로 나온다
print('=== pivot_table ===')
print(sales.pivot_table(index='year', columns='type', values='revenue', aggfunc='sum'))

print()
# aggfunc 로 집계 방법 지정, fill_value 로 빈 칸(NaN)을 0 으로 채운다
pt = sales.pivot_table(index='year', columns='type', values='title', aggfunc='count', fill_value=0)
print('연도별 종류별 작품 수:')
print(pt)
# 이 표는 그대로 그래프로 그릴 수 있다 -> pt.plot(kind='bar')

> **종합 실습 참고**: 원본 08번 노트북은 이어서 `pd.read_csv('../data/netflix_titles.csv')`로 넷플릭스 데이터를 불러와 `.str.split().explode()` → `value_counts()` → `pivot_table()`까지 종합 실습을 합니다. 데이터 파일이 있어야 실행되므로 이 통합본에서는 생략했습니다.

## 2. Numpy

`numpy`는 **숫자 계산 전용 라이브러리**입니다. pandas의 DataFrame은 사실 **numpy 배열 위에 이름표를 붙인 것**이라, numpy를 알면 pandas가 왜 그렇게 동작하는지 이해됩니다.

### 2-1. ndarray — 리스트와 무엇이 다른가

- **리스트**: 아무 타입이나 담을 수 있음. 계산하려면 반복문 필요
- **ndarray**: **같은 타입만** 담음. 대신 **전체를 한 번에 계산**

In [ ]:
py_list = [1, 2, 3, 4, 5]
arr = np.array(py_list)

print('리스트 :', py_list, type(py_list))
print('배열   :', arr, type(arr))

print()
print('shape (모양) :', arr.shape)    # (5,) -> 1차원 5개
print('dtype (타입) :', arr.dtype)
print('ndim  (차원) :', arr.ndim)

In [ ]:
# 2차원 배열 = 표. DataFrame 의 실체가 이것이다
mat = np.array([
    [1, 2, 3],
    [4, 5, 6],
])
print(mat)
print('shape:', mat.shape, '-> 2행 3열')

print()
# 자주 쓰는 생성 함수
print('arange(0,10,2) :', np.arange(0, 10, 2))     # 0부터 10 전까지 2씩
print('zeros(3)       :', np.zeros(3))              # 0 으로 채운 배열
print('linspace(0,1,5):', np.linspace(0, 1, 5))     # 0~1 을 5등분

### 2-2. 벡터 연산 — 반복문 없이 계산

numpy의 핵심입니다. **배열에 연산자를 쓰면 모든 원소에 한 번에 적용**됩니다. `df['price'] * 1.1`이 동작하는 이유가 바로 이것입니다.

In [ ]:
prices = np.array([25000, 30000, 28000, 33000])

# 리스트였다면 반복문이 필요하다
py_result = [p * 1.1 for p in [25000, 30000, 28000, 33000]]
print('리스트 + 반복문:', py_result)

# 배열은 그냥 곱하면 된다
print('배열 벡터 연산 :', prices * 1.1)

print()
print('덧셈  :', prices + 1000)
print('나눗셈:', prices / 1000)
print('비교  :', prices > 28000)   # 결과도 배열 (True/False)

In [ ]:
# 배열끼리의 연산은 '같은 자리끼리' 계산된다
qty = np.array([2, 1, 3, 1])

print('가격:', prices)
print('수량:', qty)
print('금액:', prices * qty)        # 자리별 곱셈
print('총액:', (prices * qty).sum())

### 2-3. 브로드캐스팅

모양이 다른 배열끼리 연산할 때, numpy가 **자동으로 모양을 맞춰줍니다.** `prices * 1.1`에서 `1.1`이 4개로 늘어난 것도 브로드캐스팅입니다.

In [ ]:
mat = np.array([
    [1, 2, 3],
    [4, 5, 6],
])

# (2,3) 배열 + 스칼라 -> 모든 원소에 더해짐
print('mat + 10:')
print(mat + 10)

print()
# (2,3) 배열 + (3,) 배열 -> 각 행에 더해짐
row = np.array([100, 200, 300])
print('mat + [100,200,300]:')
print(mat + row)

### 2-4. 불리언 마스크 — pandas 필터링의 원리

`df[df['price'] > 28000]`이 왜 동작하는지 여기서 알 수 있습니다.

1. 비교 연산 → **True/False 배열**이 만들어짐
2. 그 배열을 대괄호에 넣으면 → **True 자리의 값만** 남음

In [ ]:
prices = np.array([25000, 30000, 28000, 33000])

# 1단계: 조건 -> True/False 배열
mask = prices > 28000
print('마스크:', mask)

# 2단계: 마스크로 골라내기
print('선택된 값:', prices[mask])

print()
print('한 줄로  :', prices[prices > 28000])

print()
# 조건 결합: & (그리고) | (또는) - 각 조건을 반드시 괄호로 감쌀 것
print('28000 초과 & 33000 미만:', prices[(prices > 28000) & (prices < 33000)])

In [ ]:
# np.where(조건, 참일때, 거짓일때) - 조건에 따라 값 바꾸기
grade = np.where(prices >= 30000, '고가', '저가')
print('가격:', prices)
print('등급:', grade)

# pandas 에서도 똑같이 쓴다
df = pd.DataFrame({'price': prices})
df['grade'] = np.where(df['price'] >= 30000, '고가', '저가')
print()
print(df)

### 2-5. 집계와 `axis`

2차원 배열에서 `axis`는 **어느 방향으로 계산할지** 지정합니다.

| axis | 방향 | 결과 |
|---|---|---|
| `axis=0` | **세로**(행을 따라 내려가며) | 열별 결과 |
| `axis=1` | **가로**(열을 따라 옆으로) | 행별 결과 |

> 헷갈릴 때: **"axis=0은 행이 사라진다, axis=1은 열이 사라진다"**로 기억

In [ ]:
scores = np.array([
    [90, 80, 70],    # 학생1 의 국영수
    [60, 95, 85],    # 학생2
])
print(scores)
print('shape:', scores.shape)

print()
print('전체 평균        :', scores.mean())
print('axis=0 (과목별)  :', scores.mean(axis=0))   # 열별 -> 과목 평균
print('axis=1 (학생별)  :', scores.mean(axis=1))   # 행별 -> 학생 평균

In [ ]:
# 자주 쓰는 집계 함수
print('합계:', scores.sum())
print('최대:', scores.max(), '/ 최소:', scores.min())
print('표준편차:', round(scores.std(), 2))

print()
# argmax: '가장 큰 값' 이 아니라 '가장 큰 값의 위치(인덱스)'
print('학생별 최고점 과목 위치:', scores.argmax(axis=1))
print('-> 0=국어, 1=영어, 2=수학')

### 2-6. 결측치 `np.nan`

`np.nan`은 "값이 없음"을 뜻하는 특별한 숫자입니다. pandas의 빈 칸이 바로 이것입니다.

**주의: `nan`이 하나라도 섞이면 일반 집계 결과가 전부 `nan`이 됩니다.**

In [ ]:
heights = np.array([172, np.nan, 168, 180])
print('배열:', heights)

print()
print('mean()    :', heights.mean())       # nan! 하나만 있어도 전체가 nan
print('nanmean() :', np.nanmean(heights))  # nan 을 무시하고 계산

print()
# nan 은 자기 자신과도 같지 않다 -> == 로 못 찾는다
print('nan == nan :', np.nan == np.nan)
print('isnan()    :', np.isnan(heights))   # 이 함수로 찾아야 한다

### 2-7. pandas와의 관계

DataFrame의 각 열은 **ndarray에 인덱스와 이름을 붙인 것**입니다. 그래서 pandas에서 numpy 함수를 그대로 쓸 수 있습니다.

In [ ]:
df = pd.DataFrame({
    'name':   ['A', 'B', 'C', 'D'],
    'height': [172, 165, 168, 180],
    'weight': [55, 50, 58, 70],
})

# .values 또는 .to_numpy() 로 numpy 배열을 꺼낼 수 있다
print('Series 의 내부:', df['height'].to_numpy(), type(df['height'].to_numpy()))

print()
# numpy 함수를 pandas 열에 그대로 적용
df['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(1)
df['grade'] = np.where(df['bmi'] >= 22, '높음', '보통')
print(df)

> **종합 실습 참고**: 원본 09번 노트북은 `pd.read_csv('../data/인구현황.csv')`로 실제 인구 데이터에 벡터 연산을 적용하는 실습이 이어집니다. 데이터 파일이 있어야 실행되므로 생략했습니다.

## 3. 날짜와 시간 다루기

스크래핑한 날짜는 대부분 **문자열**입니다. `"26/08/2014"`는 문자열일 뿐이라 뺄셈도, 연도 추출도, 정렬도 되지 않습니다. **날짜형으로 바꾸는 순간** 이 모든 것이 가능해집니다.

### 3-1. datetime / timedelta 기본

- `date`: 날짜만 (연월일)
- `datetime`: 날짜 + 시각
- `timedelta`: **기간** (두 날짜의 차이, 또는 더하고 뺄 기간)

In [ ]:
from datetime import datetime, date, timedelta

d1 = date(2014, 8, 26)
d2 = date(2020, 1, 1)

# 날짜끼리 빼면 timedelta(기간) 이 나온다
gap = d2 - d1
print('두 날짜의 차이:', gap)
print('일수만:', gap.days)

print()
# 날짜 + 기간 = 새로운 날짜
print('100일 후:', d1 + timedelta(days=100))
print('1주 전  :', d1 - timedelta(weeks=1))

print()
# 개별 요소 꺼내기
print('연:', d1.year, '/ 월:', d1.month, '/ 일:', d1.day)
print('요일 번호:', d1.weekday(), '(0=월요일, 6=일요일)')

### 3-2. 문자열 ↔ 날짜 (strptime / strftime)

이름이 헷갈리기 쉬운데, 마지막 글자로 기억하면 됩니다.

- `strptime`: **p = parse** → 문자열을 **읽어서** 날짜로
- `strftime`: **f = format** → 날짜를 **꾸며서** 문자열로

| 기호 | 의미 | 예 |
|---|---|---|
| `%Y` | 연도 4자리 | 2014 |
| `%m` | 월 2자리 | 08 |
| `%d` | 일 2자리 | 26 |
| `%H:%M:%S` | 시:분:초 | 14:30:00 |

In [ ]:
# 문자열 -> 날짜 : 문자열의 '생김새' 를 형식으로 알려줘야 한다
s = '26/08/2014'
parsed = datetime.strptime(s, '%d/%m/%Y')    # 일/월/연 순서
print('문자열:', s, type(s))
print('날짜형:', parsed, type(parsed))

print()
# 날짜 -> 문자열 : 원하는 모양으로 출력
print('%Y-%m-%d      :', parsed.strftime('%Y-%m-%d'))
print('%Y년 %m월 %d일 :', parsed.strftime('%Y년 %m월 %d일'))
print('%y/%m         :', parsed.strftime('%y/%m'))

In [ ]:
# 형식이 맞지 않으면 에러가 난다 -> 스크래핑 데이터에서 매우 흔한 상황
bad = '2014-08-26'
try:
    datetime.strptime(bad, '%d/%m/%Y')
except ValueError as exp:
    print('에러 발생:', exp)
    print('-> 문자열 생김새와 형식 문자열이 반드시 일치해야 한다')

### 3-3. pandas `to_datetime` — 열 전체를 한 번에

`strptime`은 값 하나씩 처리합니다. 표의 열 전체는 `pd.to_datetime()`으로 한 번에 바꿉니다.

| 옵션 | 하는 일 |
|---|---|
| `format='%d/%m/%Y'` | 형식을 명시 (가장 빠르고 안전) |
| `dayfirst=True` | 일/월/연 순서임을 알림 |
| `errors='coerce'` | 변환 실패한 값을 에러 대신 **NaT**(빈 날짜)로 |

In [ ]:
df = pd.DataFrame({
    'name':  ['A', 'B', 'C', 'D'],
    'debut': ['26/08/2014', '31/10/2015', '11/10/2017', '이상한값'],
})

print('변환 전 dtype:', df['debut'].dtype)   # object = 문자열

# errors='coerce' : 변환 못 하는 값은 NaT 로 만들고 계속 진행
df['debut'] = pd.to_datetime(df['debut'], dayfirst=True, errors='coerce')

print('변환 후 dtype:', df['debut'].dtype)   # datetime64[ns]
print()
print(df)
print('변환 실패(NaT) 개수:', df['debut'].isna().sum())

In [ ]:
# 날짜형이 되면 '비교' 와 '정렬' 이 제대로 동작한다
print('2015년 이후 데뷔:')
print(df[df['debut'] >= '2015-01-01'])

print()
print('데뷔일 순 정렬:')
print(df.sort_values('debut'))

# 문자열이었다면 '31/10/2015' < '26/08/2014' 처럼 글자 순서로 비교되어 엉뚱한 결과가 나온다

### 3-4. `.dt` 접근자 — 연·월·요일 추출

문자열 열에 `.str`을 붙였듯이, 날짜 열에는 `.dt`를 붙입니다.

In [ ]:
df2 = df.dropna(subset=['debut']).copy()    # NaT 행 제외

df2['year']    = df2['debut'].dt.year
df2['month']   = df2['debut'].dt.month
df2['weekday'] = df2['debut'].dt.day_name()      # 요일 이름
df2['ym']      = df2['debut'].dt.strftime('%Y-%m')   # 원하는 문자열로

print(df2)
print()
print('사용 가능한 것들: .dt.year .dt.month .dt.day .dt.hour')
print('                 .dt.dayofweek .dt.day_name() .dt.quarter .dt.strftime()')

### 3-5. 나이 계산 — 튜플 비교 활용

"생일이 지났는지"를 따져야 해서 단순히 연도만 빼면 틀립니다.

```
나이 = 올해 - 태어난해 - (생일이 아직 안 지났으면 1)
```

`(month, day)`를 튜플로 묶어 비교하면 월·일을 한 번에 비교할 수 있습니다.

In [ ]:
# 튜플 비교 복습: 앞 요소부터 차례로 비교한다
print('(12, 15) < (1, 1)  :', (12, 15) < (1, 1))    # 12 > 1 이므로 False
print('(12, 15) < (12, 17):', (12, 15) < (12, 17))  # 12==12 -> 15 < 17 이므로 True

print()
print('-> True(1) / False(0) 는 숫자처럼 뺄셈에 쓸 수 있다')
print('   True 는', int(True), ', False 는', int(False))

In [ ]:
def calc_age(birth, today=None):
    # 만 나이를 계산한다. today 를 생략하면 오늘 날짜 기준.
    today = today or date.today()
    # (월, 일) 튜플 비교로 '생일이 아직 안 지났는지' 판정
    not_yet = (today.month, today.day) < (birth.month, birth.day)
    return today.year - birth.year - not_yet


기준일 = date(2024, 6, 15)
print('1997-09-10 생 ->', calc_age(date(1997, 9, 10), 기준일), '세  (생일 전)')
print('1997-03-10 생 ->', calc_age(date(1997, 3, 10), 기준일), '세  (생일 지남)')
print('1997-06-15 생 ->', calc_age(date(1997, 6, 15), 기준일), '세  (생일 당일)')

### 3-6. 기간별 집계 — 연도별 추이

날짜형으로 바꿔두면 **연도별·월별 집계**가 한 줄로 끝납니다.

In [ ]:
logs = pd.DataFrame({
    'date':  pd.to_datetime([
        '2023-01-15', '2023-01-20', '2023-02-03',
        '2023-02-11', '2024-01-05', '2024-03-22',
    ]),
    'views': [100, 150, 200, 120, 300, 250],
})
print(logs)

In [ ]:
# 연도별 집계 - .dt.year 로 그룹
print('연도별 합계:')
print(logs.groupby(logs['date'].dt.year)['views'].sum())

print()
# 월별 집계 - strftime 으로 'YYYY-MM' 문자열 그룹
print('월별 합계:')
print(logs.groupby(logs['date'].dt.strftime('%Y-%m'))['views'].sum())

In [ ]:
# resample: 날짜를 인덱스로 두면 'M'(월), 'Y'(연) 단위 재집계가 가능
ts = logs.set_index('date')
print('월 단위 resample:')
print(ts['views'].resample('ME').sum())
print()
print('-> 데이터가 없는 달도 0 으로 채워져 나온다는 점이 groupby 와 다르다')

> **종합 실습 참고**: 원본 10번 노트북은 `pd.read_csv('../data/kpopidolsv3.csv')`로 실제 K-Pop 아이돌 데이터를 불러와 생년월일·데뷔일 변환 → 데뷔 나이 계산 → 소속사별 평균 데뷔 나이까지 종합 실습을 합니다. 데이터 파일이 있어야 실행되므로 생략했습니다.

## 4. 인코딩과 디코딩

`encoding='cp949'`, `.encode()`, `base64`, `%EC%95%88` … 스크래핑을 하다 보면 계속 마주치지만 정확히 무슨 일이 일어나는지 알기 어려운 개념입니다.

> **목표**: "인코딩"이라는 한 단어가 서로 다른 두 가지 일을 가리킨다는 것을 구분하기

### 4-1. 컴퓨터에는 '글자'가 없다 — `str`과 `bytes`

컴퓨터가 저장하고 전송할 수 있는 것은 **숫자(바이트)뿐**입니다. `'안'`이라는 글자 자체를 저장하는 방법은 없습니다. 그래서 "어떤 글자를 어떤 숫자로 볼지" 약속표가 필요합니다. 그 약속표가 **문자 인코딩**(UTF-8, CP949 등)입니다.

| 타입 | 정체 | 표기 |
|---|---|---|
| `str` | 사람이 읽는 **글자** | `'안녕'` |
| `bytes` | 컴퓨터가 다루는 **숫자 나열** | `b'\xec\x95\x88...'` |

In [ ]:
s = '안녕'

# str : 사람이 읽는 글자
print('타입:', type(s), '/ 길이:', len(s), '글자')

# encode() : 약속표에 따라 숫자로 바꾼다
b = s.encode('utf-8')
print('타입:', type(b), '/ 길이:', len(b), '바이트')

print()
print('bytes 표기 :', b)          # b'...' 로 표시됨
print('실제 숫자  :', list(b))    # 진짜 정체는 그냥 숫자 나열이다

In [ ]:
# 같은 '안녕' 이라도 약속표가 다르면 숫자가 완전히 달라진다
utf8 = '안녕'.encode('utf-8')
cp949 = '안녕'.encode('cp949')

print('UTF-8 :', utf8, '->', list(utf8), f'({len(utf8)}바이트)')
print('CP949 :', cp949, '->', list(cp949), f'({len(cp949)}바이트)')

print()
print('-> UTF-8 은 한글 1글자를 3바이트, CP949 는 2바이트로 표현한다')
print('-> 그래서 같은 글자인데도 파일 크기가 달라진다')

### 4-2. encode / decode 기본

| 용어 | 방향 | 한 문장 |
|---|---|---|
| **인코딩** `encode()` | `str` → `bytes` | 약속표에 따라 **숫자로 바꾸기** |
| **디코딩** `decode()` | `bytes` → `str` | 같은 약속표로 **다시 글자로 읽기** |

> **가장 중요한 원리**: 인코딩과 디코딩은 **반드시 짝**이다. 넣을 때 쓴 규칙과 꺼낼 때 쓴 규칙이 같아야만 원본이 나온다.

In [ ]:
원본 = '파이썬'

# 왕복: encode -> decode
숫자 = 원본.encode('utf-8')     # str  -> bytes
복원 = 숫자.decode('utf-8')     # bytes -> str

print('원본:', 원본)
print('숫자:', 숫자)
print('복원:', 복원)
print()
print('원본과 같은가?', 원본 == 복원)

# 인자를 생략하면 UTF-8 이 기본값 (파이썬 3 기준)
print()
print(".encode() 기본값은 utf-8:", '가'.encode() == '가'.encode('utf-8'))

### 4-3. 규칙이 어긋나면 — 글자 깨짐의 정체

실무에서 겪는 한글 깨짐은 파일이 손상된 것이 아닙니다. **숫자는 멀쩡한데, 읽는 약속표를 잘못 골랐을 뿐**입니다.

| 유형 | 결과 | 위험도 |
|---|---|---|
| 해석 불가능한 바이트 | `UnicodeDecodeError` 에러 발생 | 낮음 (바로 알아챔) |
| 우연히 해석 가능한 바이트 | 조용히 이상한 글자로 변환 | **높음** (모르고 지나감) |

In [ ]:
b = '안녕'.encode('utf-8')       # UTF-8 규칙으로 숫자화
print('원본 바이트:', b)

print()
# 유형 A: 같은 규칙 -> 정상
print("utf-8 로 디코딩 :", b.decode('utf-8'), ' <- 정상')

print()
# 유형 B: 해석 불가능 -> 에러 (그나마 다행인 경우)
try:
    b.decode('cp949')
except UnicodeDecodeError as exp:
    print('cp949 로 디코딩 : UnicodeDecodeError')
    print('   이유:', exp.reason)

In [ ]:
# errors 옵션: 깨진 글자를 만났을 때의 처리 방법
b2 = '안녕hello'.encode('utf-8')

print("errors='strict'  (기본) -> 에러 발생")
try:
    b2.decode('ascii')
except UnicodeDecodeError:
    print('   UnicodeDecodeError')

# 읽을 수 없는 바이트를 버리거나 대체 문자로 바꾼다
print("errors='ignore'  :", repr(b2.decode('ascii', errors='ignore')))
print("errors='replace' :", repr(b2.decode('ascii', errors='replace')))

print()
print('-> ignore/replace 는 데이터가 손실된다. 원인을 모를 때 임시로만 쓸 것')

### 4-4. `repr()` — 보이지 않는 문자를 찾아내기

| 함수 | 대상 | 목적 |
|---|---|---|
| `print()` / `str()` | 사람 | 보기 좋게 |
| `repr()` | 개발자 | **정확하게** (따옴표·이스케이프 그대로) |

깨진 문자열에는 화면에 표시되지 않는 제어문자가 섞여 있어서, 그냥 `print`하면 무슨 값인지 알 수 없습니다. f-string 안에서는 `{값!r}`이 `repr(값)`의 축약 표기입니다.

In [ ]:
samples = [
    ('앞뒤 공백', '  파이썬  '),
    ('개행 포함', '줄1\n줄2'),
    ('탭 포함', '이름\t점수'),
    ('빈 문자열', ''),
]

for label, value in samples:
    print(f'[{label}]')
    print(f'  print() : {value}')       # 그냥 출력 - 숨은 문자가 안 보인다
    print(f'  repr()  : {value!r}')     # repr 출력 - 숨은 문자가 드러난다
    print(f'  글자 수 : {len(value)}')
    print('-' * 42)

In [ ]:
# 실전 - '눈으로는 같은데 키가 안 맞는' 상황을 repr 로 진단하기
# BOM(﻿)이 '이름' 앞에 숨어있는 딕셔너리라고 가정
data = {'﻿이름': '홍길동', '나이 ': 20}

print('조회 시도:')
print("  data.get('이름') ->", data.get('이름'), '<- None!')

print()
print('repr 로 보면 원인이 드러난다:')
for k in data:
    print(f'  {k!r}  (길이 {len(k)})')

print()
# 원인을 알았으면 정리한다 (lstrip 으로 BOM 문자를 직접 지정해 제거)
cleaned = {k.lstrip('﻿').strip(): v for k, v in data.items()}
print('정리 후 키:', [repr(k) for k in cleaned])
print("cleaned['이름'] ->", cleaned['이름'])

### 4-5. 인코딩의 두 층위

같은 "인코딩"이라는 단어가 서로 다른 층위에서 쓰입니다.

| | 층위 1 — 문자 인코딩 | 층위 2 — 표현 변환 |
|---|---|---|
| 무엇을 바꾸나 | **글자 ↔ 숫자** | **숫자 ↔ 숫자** |
| 타입 | `str` ↔ `bytes` | `bytes` ↔ `bytes` |
| 파이썬 | `.encode()` / `.decode()` | `base64.b64encode()` / `quote()` |
| 예 | UTF-8, CP949, EUC-KR | base64, URL 인코딩, hex |
| 왜 필요한가 | 글자를 저장·전송하려고 | **못 지나가는 통로를 통과시키려고** |

In [ ]:
import base64

s = '안녕'

step1 = s.encode('utf-8')          # (1) 층위1 : str  -> bytes
step2 = base64.b64encode(step1)    # (2) 층위2 : bytes -> bytes
step3 = step2.decode('utf-8')      # (3) 층위1 : bytes -> str

print('(1) .encode()   :', repr(s), '->', repr(step1))
print('(2) b64encode() :', repr(step1), '->', repr(step2))
print('(3) .decode()   :', repr(step2), '->', repr(step3))

print()
print('타입 변화 : str -> bytes -> bytes -> str')

### 4-6. base64 — 표현 변환은 왜 필요한가

메일 본문, JSON, URL 같은 통로는 **글자로 표현 가능한 것만** 지나갈 수 있습니다. 그런데 이미지 같은 바이너리에는 글자로 표현할 수 없는 바이트가 섞여 있습니다. 그래서 **어디서나 안전한 64개 문자**(`A-Z a-z 0-9 + /`)만으로 다시 표현하는 것이 base64입니다.

- **장점**: 어떤 통로든 안전하게 통과
- **대가**: 길이가 약 **4/3배**로 늘어남

In [ ]:
data = '안녕'.encode('utf-8')
encoded = base64.b64encode(data)

print('원본 바이트 :', data, f'({len(data)}바이트)')
print('base64      :', encoded, f'({len(encoded)}바이트)')
print(f'길이 비율   : {len(encoded) / len(data):.2f}배')

print()
# 되돌리기는 정확히 역순
back = base64.b64decode(encoded).decode('utf-8')
print('복원:', back, '/ 원본과 같은가?', back == '안녕')

In [ ]:
# 실전 예: 이미지를 HTML 에 직접 박아넣기 (data URI)
png_bytes = bytes([0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A])   # PNG 시그니처

# 이 바이트들은 글자로 표현할 수 없다 -> 그대로 HTML 에 넣을 수 없다
print('원본 바이트:', png_bytes)

# base64 로 바꾸면 문자열이 되어 HTML 에 넣을 수 있다
b64_str = base64.b64encode(png_bytes).decode()
print('base64     :', b64_str)
print()
print(f'<img src="data:image/png;base64,{b64_str}">')

### 4-7. URL 인코딩 — 네이버 API 검색어

URL에는 **한글, 공백, 특수문자를 그대로 넣을 수 없습니다.** `?query=파이썬 기초`처럼 보내면 공백에서 주소가 끊깁니다. 그래서 **안전하지 않은 바이트를 `%XX`(16진수)로 바꾸는** 것이 URL 인코딩입니다. 이것도 **층위 2**입니다.

In [ ]:
from urllib.parse import quote, unquote, urlencode

keyword = '파이썬 기초'

# quote() : 한글·공백을 %XX 형태로 변환
encoded_kw = quote(keyword)
print('원본     :', keyword)
print('URL 인코딩:', encoded_kw)

print()
print('되돌리기 :', unquote(encoded_kw))

In [ ]:
# 실전: 네이버 검색 API 요청 URL 만들기
BASE = 'https://openapi.naver.com/v1/search/news.json'

# 방법 1) 직접 조립 - 반드시 quote() 를 거쳐야 한다
url1 = f'{BASE}?query={quote("파이썬")}&display=10'
print('직접 조립:', url1)

# 방법 2) urlencode() - 여러 파라미터를 한 번에 (권장)
params = {'query': '파이썬 기초', 'display': 10, 'sort': 'sim'}
url2 = f'{BASE}?{urlencode(params)}'
print('urlencode:', url2)

print()
# 방법 3) requests 는 params= 로 넘기면 알아서 인코딩해준다 (가장 편함)
print('requests 사용 시: requests.get(BASE, params=params)')
print('  -> 내부에서 자동으로 urlencode 를 수행하므로 quote() 를 직접 부를 필요 없다')

### 4-8. 실전 ① CSV 파일 인코딩

**`pd.read_csv()`의 기본 인코딩은 UTF-8**입니다. 그런데 국내 공공데이터·엑셀에서 내려받은 CSV는 대부분 **CP949**(=EUC-KR 확장)입니다. 파일이 잘못된 게 아니라 읽는 약속표를 안 알려준 것뿐입니다.

In [ ]:
import io

# CP949 로 저장된 CSV 를 흉내낸다 (실제 파일 대신 메모리에서)
csv_text = '이름,점수\n홍길동,90\n김철수,85\n'
cp949_bytes = csv_text.encode('cp949')

print('파일에 저장된 실제 내용(바이트):', cp949_bytes[:20], '...')

print()
# 잘못된 인코딩으로 읽으면?
try:
    pd.read_csv(io.BytesIO(cp949_bytes))          # 기본값 utf-8
except UnicodeDecodeError as exp:
    print('encoding 지정 안 함 -> UnicodeDecodeError')
    print('   이유:', exp.reason)

In [ ]:
# 올바른 인코딩을 알려주면 정상적으로 읽힌다
df_csv = pd.read_csv(io.BytesIO(cp949_bytes), encoding='cp949')
print(df_csv)

print()
print('=== 인코딩을 모를 때의 대처 순서 ===')
print('1) encoding="utf-8"      : 요즘 만들어진 파일 대부분')
print('2) encoding="cp949"      : 국내 공공데이터/엑셀 저장 파일')
print('3) encoding="utf-8-sig"  : 엑셀이 만든 UTF-8 (BOM 이 붙어있음)')
print('4) 그래도 안 되면 encoding="latin-1" 로 일단 읽어 내용 확인')

In [ ]:
# utf-8-sig : 엑셀이 붙이는 BOM(Byte Order Mark) 처리
with_bom = '이름,점수\n홍길동,90\n'.encode('utf-8-sig')

print('BOM 포함 바이트 앞부분:', with_bom[:6])
print('-> 앞의 3바이트가 BOM')

print()
# pandas 는 BOM 을 알아서 걸러준다 -> read_csv 에서는 문제가 잘 드러나지 않는다
print("pandas encoding='utf-8'     :", list(pd.read_csv(io.BytesIO(with_bom), encoding='utf-8').columns))
print("pandas encoding='utf-8-sig' :", list(pd.read_csv(io.BytesIO(with_bom), encoding='utf-8-sig').columns))
print('-> pandas 는 둘 다 정상이다 (BOM 자동 제거)')

print()
# 문제는 '직접 decode' 할 때 드러난다
print("decode('utf-8')     :", repr(with_bom.decode('utf-8')[:6]), ' <- BOM(\\ufeff) 이 남아있다')
print("decode('utf-8-sig') :", repr(with_bom.decode('utf-8-sig')[:6]), ' <- 깨끗하다')

### 4-9. 실전 ② requests의 `.text` vs `.content`

웹 응답도 결국 바이트로 도착합니다. 이 둘의 차이를 알면 한글 깨짐을 스스로 고칠 수 있습니다.

| 속성 | 타입 | 설명 |
|---|---|---|
| `res.content` | `bytes` | **서버가 보낸 원본 바이트** (가공 전) |
| `res.text` | `str` | `res.encoding` 규칙으로 **디코딩한 결과** |

즉 `res.text`는 **`res.content.decode(res.encoding)`**과 같습니다. `res.encoding`을 서버가 잘못 알려주면 `res.text`가 깨집니다.

In [ ]:
import requests

# 네트워크 없이 응답 객체를 직접 만들어 원리만 확인한다
res = requests.models.Response()
res._content = '한글 뉴스 제목'.encode('cp949')   # 서버가 CP949 로 보냈다고 가정
res.status_code = 200

# 서버가 인코딩을 잘못 알려준 상황
res.encoding = 'utf-8'
print("res.encoding = 'utf-8' ->", repr(res.text), ' <- 깨짐')

# 올바른 인코딩을 직접 지정하면 정상으로 돌아온다
res.encoding = 'cp949'
print("res.encoding = 'cp949' ->", repr(res.text), ' <- 정상')

print()
print('원본 바이트(content):', res.content)
print('-> content 는 항상 그대로다. text 만 encoding 에 따라 달라진다')

**실전 대처 패턴**
```python
# 방법 1) 인코딩을 직접 지정
res = requests.get(url)
res.encoding = 'cp949'      # 또는 'euc-kr'
soup = BeautifulSoup(res.text, 'html.parser')

# 방법 2) 응답 내용을 보고 자동 추정하게 하기
res.encoding = res.apparent_encoding

# 방법 3) 원본 바이트를 넘기고 파서가 알아서 판단하게 하기 (가장 무난)
soup = BeautifulSoup(res.content, 'html.parser')
```

## 5. 예상문제 (사지선다)

새로 추가된 4개 파트 대비용입니다.

**문제 1.** `pd.merge(A, B, on='key', how='left')`에 대한 설명으로 옳은 것은?
① A, B 양쪽에 모두 있는 key만 남긴다  ② A는 전부 남기고 B에 없으면 NaN으로 채운다  ③ B를 기준으로 A를 덮어쓴다  ④ 인덱스만 기준으로 합친다

정답 ②

**문제 2.** `pd.concat([df1, df2])`에서 `ignore_index=True`를 빼면 어떻게 되는가?
① 에러가 발생한다  ② 인덱스가 0,1,0,1처럼 중복될 수 있다  ③ 자동으로 정렬된다  ④ df2가 무시된다

정답 ②

**문제 3.** `"Drama, Comedy"`처럼 한 칸에 여러 값이 있는 열을 값별로 집계(`value_counts`)하려면 어떤 순서가 맞는가?
① `value_counts()` → `explode()`  ② `str.split()` → `explode()` → `value_counts()`  ③ `explode()` → `str.split()`  ④ 바로 `value_counts()` 호출

정답 ②

**문제 4.** `DataFrame.apply(함수, axis=1)`에 대한 설명으로 옳은 것은?
① 각 열에 함수를 적용한다  ② 각 값 하나하나에 함수를 적용한다  ③ 각 행 전체를 Series로 받아 여러 열을 함께 사용할 수 있다  ④ 반드시 lambda만 사용 가능하다

정답 ③

**문제 5.** numpy 배열에서 `arr.mean(axis=0)`은 무엇을 계산하는가? (2차원 배열 기준)
① 전체 평균  ② 행별 평균  ③ 열별 평균  ④ 대각선 평균

정답 ③ — axis=0은 세로 방향(행이 사라짐) → 열별 결과.

**문제 6.** 다음 중 numpy에서 조건을 결합할 때 올바른 문법은?
① `arr[arr>10 and arr<20]`  ② `arr[(arr>10) & (arr<20)]`  ③ `arr[arr>10 or arr<20]`  ④ `arr[arr>10, arr<20]`

정답 ② — `and`/`or`가 아니라 `&`/`|`, 각 조건은 괄호로 감싸야 함.

**문제 7.** `np.nan`에 대한 설명으로 옳지 않은 것은?
① `np.nan == np.nan`은 True다  ② nan이 하나 섞이면 `mean()` 결과가 전체 nan이 된다  ③ `np.isnan()`으로 찾아야 한다  ④ `np.nanmean()`은 nan을 무시하고 계산한다

정답 ① — `np.nan == np.nan`은 **False**다.

**문제 8.** 문자열 `'26/08/2014'`(dd/mm/yyyy)를 datetime 객체로 변환하는 코드는?
① `datetime.strftime('26/08/2014', '%d/%m/%Y')`  ② `datetime.strptime('26/08/2014', '%d/%m/%Y')`  ③ `pd.to_datetime('26/08/2014')` (옵션 없이)  ④ `date.today()`

정답 ② — strptime(parse)이 문자열→날짜. ③은 dayfirst 옵션 없이는 월/일이 뒤바뀔 위험이 있음.

**문제 9.** `pd.to_datetime(df['컬럼'], errors='coerce')`에서 `errors='coerce'`의 역할은?
① 변환 실패 시 프로그램을 즉시 종료한다  ② 변환 실패한 값을 NaT로 만들고 계속 진행한다  ③ 모든 값을 강제로 오늘 날짜로 바꾼다  ④ 에러 로그를 파일로 저장한다

정답 ②

**문제 10.** `encode()`와 `decode()`의 관계로 옳은 것은?
① 서로 상관없이 아무 인코딩이나 조합 가능하다  ② 넣을 때 쓴 인코딩 규칙과 꺼낼 때 쓴 규칙이 같아야 원본이 복원된다  ③ decode는 str을 bytes로 바꾼다  ④ encode는 bytes를 str로 바꾼다

정답 ② — ③④는 방향이 반대로 설명됨 (encode: str→bytes, decode: bytes→str).

**문제 11.** base64 인코딩에 대한 설명으로 옳지 않은 것은?
① 바이너리 데이터를 안전한 문자로 표현하는 방법이다  ② 인코딩 결과의 길이는 원본보다 커진다  ③ 비밀번호를 안전하게 숨기는 암호화 방법이다  ④ 이미지를 HTML에 `data:image/...;base64,...` 형태로 넣을 때 쓰인다

정답 ③ — base64는 누구나 되돌릴 수 있는 인코딩이지 암호화가 아니다.

**문제 12.** 국내 공공데이터에서 내려받은 CSV 파일을 읽을 때 한글이 깨진다면 가장 먼저 시도해볼 것은?
① `pd.read_csv(파일)`을 그대로 재실행한다  ② `encoding='cp949'` 옵션을 추가한다  ③ 파일을 삭제하고 다시 받는다  ④ CSV를 JSON으로 바꾼다

정답 ②

**문제 13.** `requests`로 받은 응답의 한글이 깨질 때, 가장 무난한 해결 방법은?
① `res.text`를 그대로 사용한다  ② `res.encoding`을 무시한다  ③ `BeautifulSoup(res.content, 'html.parser')`처럼 원본 바이트를 그대로 넘긴다  ④ 재요청을 무한 반복한다

정답 ③


## 요약 — 가장 헷갈리기 쉬운 5가지

1. **`merge`(옆으로, 열 합치기) vs `concat`(아래로, 행 이어붙이기)** — 방향이 반대.
2. **`axis=0`(열별 결과) vs `axis=1`(행별 결과)** — "그 축이 사라진다"고 암기.
3. **`strptime`(문자열→날짜, parse) vs `strftime`(날짜→문자열, format)** — 이름 끝 글자(p/f)로 구분.
4. **`encode()`(str→bytes) vs `decode()`(bytes→str)** — 방향을 반대로 알면 완전히 틀린 답이 됨.
5. **인코딩(층위1: 글자↔숫자, UTF-8/CP949) vs 표현변환(층위2: 숫자↔숫자, base64/URL인코딩)** — 같은 "인코딩"이라는 말이 가리키는 두 가지 다른 개념.
